# Taller Hugging Face: fine-tuning de un LLM pequeño con LoRA

## 1. Entorno

In [26]:
#%pip install -q transformers datasets peft accelerate matplotlib bitsandbytes # Para ejecurtarlo en colab

In [27]:
# Para ejecutarlo en local
#%pip install -q -U transformers torch datasets peft accelerate pandas bitsandbytes

In [28]:
import gc
import re

import pandas as pd
import torch
from transformers import set_seed

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_ID = "Bellido/cafeterias_uam_2026"
ADAPTER_DIR = "lora-cafeterias-uam"
ADAPTER_REPO = "Bellido/qwen2.5-0.5b-cafeterias-uam-lora"
ENTRENAR = False
EPOCHS = 4
SEED = 42
set_seed(SEED)

# Detectamos el hardware primero
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16 # Filtramos por la capacidad de la GPU para usar bfloat16
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float32
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

CAN_QUANTIZE = DEVICE == "cuda" or DEVICE == "mps"
print(f"Dispositivo: {DEVICE} | dtype: {DTYPE} | cuantización 4 bits: {CAN_QUANTIZE}")

Dispositivo: mps | dtype: torch.float32 | cuantización 4 bits: True


## 2. Dataset desde el Hub

`load_dataset` descarga `Bellido/cafeterias_uam_2026` (columnas `question` y `answer`, solo split `train`).
Reservamos un 5 % para medir el *loss* en datos que el modelo no ve durante el entrenamiento.

In [29]:
from datasets import load_dataset

ds = load_dataset(DATASET_ID, split="train") # Cargamos el dataset completo, que es pequeño y cabe en memoria
print(ds)
display(ds.select(range(5)).to_pandas())

splits = ds.train_test_split(test_size=0.05, seed=SEED) # Dividimos en train/test (95%/5%)
train_ds, eval_ds = splits["train"], splits["test"]
print(f"train: {len(train_ds)} ejemplos | eval: {len(eval_ds)} ejemplos")

Dataset({
    features: ['question', 'answer'],
    num_rows: 1000
})


,question,answer
0,¿Cuánto cuesta café solo?,"El café solo cuesta 1,10 €."
1,¿Cuál es el precio de café solo?,"El café solo cuesta 1,10 €."
2,¿A cuánto está café solo?,"El café solo cuesta 1,10 €."
3,¿Cuánto vale café solo?,"El café solo cuesta 1,10 €."
4,Dime el precio de café solo,"El café solo cuesta 1,10 €."


train: 950 ejemplos | eval: 50 ejemplos


In [30]:
# Cargar el dataset desde el archivo JSON
# ds = load_dataset('json', data_files='./datasets/dataset_cafeteria_uam_2026.jsonl', split='train')
# print(ds)
# display(ds.select(range(5)).to_pandas())

# splits = ds.train_test_split(test_size=0.05, seed=SEED)
# train_ds, eval_ds = splits['train'], splits['test']
# print(f"train: {len(train_ds)} ejemplos | eval: {len(eval_ds)} ejemplos")

## 3. Modelo base y prueba *antes* de entrenar

Usamos un modelo *instruct* ya preparado para chat (Qwen2.5-0.5B-Instruct, ~0.5B parámetros). El tokenizador aplica su **plantilla de chat**: es importante usar la misma en entrenamiento e inferencia.

In [31]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID) # Cargamos el tokenizador del propio modelo

if tokenizer.pad_token is None: # Comporbamos si el modelo tiene token de padding, y si no lo tiene, lo asignamos al token de fin de secuencia
    # El token padding se utiliza para cuando un modelo procesa varias frases a la vez, para que tengan todas la misma longitud se usa el ultimo token de la frase como padding
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM = "Eres un asistente de las cafeterías de la Universidad Autónoma de Madrid. Responde de forma breve y precisa."


def build_messages(question, answer=None): # Crea la estructura de mensajes concreta para el modelo, con el rol de sistema, usuario y asistente
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": answer}) # Si se proporciona una respuesta, la añadimos como mensaje del asistente para crear un contexto de conversación
    return messages


@torch.inference_mode()
def ask(model, question, max_new_tokens=48): # Limitamos la respuesta a 48 tokens
    prompt = tokenizer.apply_chat_template(build_messages(question), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # Convertimos el prompt a tensores y los movemos a la GPU
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False, # determinista, para poder comparar las respuestas, es Greedy
        temperature=None, top_p=None, top_k=None, # No aplican cuando no hay aleatoriedad
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    # Recorta la pregunta original del resultado generado. Sin este recorte, el modelo devolvería tanto la pregunta como la respuesta juntas

### Vamos a cargar el modelo para probarlo antes de hacer el fine tuning

In [32]:
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE) # Descarga el modelo base y mueve los pesos a la GPU
n_params = base.num_parameters()
fp_gb = base.get_memory_footprint() / 1e9
print(f"{n_params / 1e6:.0f}M parámetros | memoria en {str(DTYPE).split('.')[-1]}: {fp_gb:.2f} GB")

TEST_QUESTIONS = [
    "¿Cuánto cuesta un café solo?",
    "¿Qué precio tiene el bocadillo de lomo?",
    "¿Cuánto vale el menú autoservicio?",
    "¿Cuánto cuesta la pizza entera?",
    "¿Cuánto cuesta el bono de 10 comidas?",
    "¿Qué opciones vegetarianas hay en la cafetería?",
]

answers_base = {q: ask(base, q) for q in TEST_QUESTIONS}
for q, a in answers_base.items():
    print(f"P: {q}\nR: {a}\n")

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 141.98it/s]


494M parámetros | memoria en float32: 1.98 GB
P: ¿Cuánto cuesta un café solo?
R: El costo del café en una cafetería de la Universidad Autónoma de Madrid puede variar dependiendo de varios factores, como el tipo de café (especialmente si es de origen local), la ubicación y el tiempo

P: ¿Qué precio tiene el bocadillo de lomo?
R: El precio del bocadillo de lomo en la cafetería de la Universidad Autónoma de Madrid es de 15 euros.

P: ¿Cuánto vale el menú autoservicio?
R: El menú autoservicio en la Universidad Autónoma de Madrid es gratuito para los estudiantes, pero se puede pagar con tarjetas de crédito o débito.

P: ¿Cuánto cuesta la pizza entera?
R: La pizzas entera en la Universidad Autónoma de Madrid puede variar dependiendo del tipo de pizza, el tamaño y los ingredientes adicionales que se incluyan. Sin embargo, generalmente, una pizza completa puede costarse

P: ¿Cuánto cuesta el bono de 10 comidas?
R: El bono de 10 comidas en una cafetería de la UAM varía dependiendo del lugar y e

### Métrica sencilla para la demo de la cafetería

Como las respuestas del dataset contienen precios (`1,05€`) vamos a medir qué fracción de respuestas generadas **contiene todos los precios de la respuesta correcta**.

Se mide sobre preguntas del *train* (¿ha memorizado los datos?) y sobre preguntas reservadas (*eval*).

In [33]:
if DATASET_ID == "Bellido/cafeterias_uam_2026":
    PRICE_RE = re.compile(r"\d+,\d{2}") # Regex para detectar precios en formato "X,XX" (euros con coma decimal)


    def prices(text):
        return set(PRICE_RE.findall(text))


    def price_accuracy(model, rows, n=20):
        # Filtra filas donde la respuesta real contiene al menos un precio y toma un máximo de 'n'
        rows = [r for r in rows if prices(r["answer"])][:n]
        
        # Cuenta cuántas veces el modelo incluyó todos los precios correctos
        hits = sum(prices(r["answer"]) <= prices(ask(model, r["question"])) for r in rows)
        
        # Devuelve la proporción de aciertos (de 0.0 a 1.0)
        return hits / len(rows)


    probe_train = train_ds.shuffle(seed=0).select(range(60))   # nos quedamos con 20 que tengan precio
    probe_eval = eval_ds.shuffle(seed=0)

    metrics = {"base": {
        "train": price_accuracy(base, probe_train),
        "eval": price_accuracy(base, probe_eval),
    }}
    print("Modelo base ->", metrics["base"])

Modelo base -> {'train': 0.0, 'eval': 0.0}


## 4. Cuantización

**Idea:** Guardar cada peso con menos bits. Un `float16` ocupa 2 bytes; un entero de 4 bits ocupa 0,5 bytes → el modelo pesa ~4 veces menos.

### Cuantización  con `bitsandbytes`

Los cálculos se hacen en 16 bits pero solo se almacenan en 4, es decir, a la hora de hacer la inferencia, el modelo se descomprime a 16 bits

Vamos a comporbar que tras la cuantización el modelo sigue siendo relativamente coherente

In [34]:
if CAN_QUANTIZE:
    from transformers import BitsAndBytesConfig

    # Liberamos memoria de la GPU antes de cargar el modelo cuantizado
    del base
    gc.collect()
    torch.cuda.empty_cache()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE, # los cálculos se hacen en 16 bits; solo se almacenan en 4, es decir, a la hora de hacer la inferencia, el modelo se descomprime a 16 bits
    )
    
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map={"": 0}) # De nuevo, cargamos el modelo base, pero esta vez con cuantización a 4 bits
    q_gb = model.get_memory_footprint() / 1e9
    print(f"Memoria: {fp_gb:.2f} GB (16 bits) -> {q_gb:.2f} GB (4 bits)") # Vemos como ahora ocupa tan solo 1/4 de la memoria que antes, y podemos cargarlo en GPUs con menos memoria

    answers_quant = {q: ask(model, q) for q in TEST_QUESTIONS}
    for q, a in answers_quant.items():
        print(f"P: {q}\nR: {a}\n")
else:
    model = base # Si no puede cuantizar usamos el modelo base sin cuantización

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 210.10it/s]


Memoria: 1.98 GB (16 bits) -> 0.45 GB (4 bits)
P: ¿Cuánto cuesta un café solo?
R: El costo del café solo puede variar dependiendo del lugar, el tipo de café (especialmente el café blanco) y otros factores como la calidad del producto. Sin embargo, en general, el café básico puede costarse entre

P: ¿Qué precio tiene el bocadillo de lomo?
R: El precio del bocadillo de lomo en la cafetería de la Universidad Autónoma de Madrid es de 10 euros.

P: ¿Cuánto vale el menú autoservicio?
R: El menú autoservicio es una opción popular en las cafeterías de la Universidad Autónoma de Madrid, pero no tiene un valor monetario. Es simplemente una opción que los clientes pueden elegir según su preferencia.

P: ¿Cuánto cuesta la pizza entera?
R: La pizza entera, incluyendo el sabor, es costosa según la región del país donde se produjo. Sin embargo, no puedo proporcionar información específica sobre cuánta costará una pizza entera en cada región debido

P: ¿Cuánto cuesta el bono de 10 comidas?
R: El costo

## 5. LoRA

En vez de actualizar la matriz de pesos `W` (d × d), la "**congelamos**" y aplicamlos LoRA:

`W' = W + (alpha / r) · B · A`   con   `A: r × d`,  `B: d × r`,  `r << d`

- Solo se entrenan `A` y `B` (millones de parámetros menos). `B` empieza en 0, así que al principio el modelo es idéntico al original.
- `r` (rango) controla la capacidad del adaptador (el rango); `alpha` escala cuánto contribuye la nueva matriz.
- Con el modelo base cuantizado a 4 bits esto es **QLoRA**: congelamos los pesos en 4 bits, y los adaptedores LoRA en 16/32 bits.

In [35]:
from peft import PeftModel

if ENTRENAR:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    if CAN_QUANTIZE:
        # Prepara el modelo cuantizado para entrenar
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

    lora_config = LoraConfig(
        r=16, # Usamos un rango de 16 para la matriz de bajo rango, lo que significa que la matriz de pesos original se aproximará mediante dos matrices más pequeñas de tamaño 16
        lora_alpha=32,
        lora_dropout=0.05, # Evita overfitting descartando aleatoriamente algunas conexiones durante el entrenamiento
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Especifica qué capas del modelo se adaptarán mediante LoRA, son las más importantes para el aprendizaje de representaciones
        task_type="CAUSAL_LM", # Causal LM significa que el modelo predice la siguiente palabra en una secuencia estándar para tareas de generación de texto
    )
    model = get_peft_model(model, lora_config) # Crea un modelo LoRA a partir del modelo base y la configuración de LoRA
    
else: # Si no queremos entrenar, cargamos el adaptador LoRA ya entrenado desde el repositorio
    model = PeftModel.from_pretrained(model, ADAPTER_REPO)
    print(f"Adaptador cargado desde {ADAPTER_REPO}")
    
model.print_trainable_parameters()

# Una capa concreta
layer = model.base_model.model.model.layers[0].self_attn.q_proj
A, B = layer.lora_A["default"].weight, layer.lora_B["default"].weight
full = layer.base_layer.weight.numel()
print(f"\nq_proj capa 0: W {tuple(layer.base_layer.weight.shape)} = {full:,} pesos (congelados)")
print(f"  A {tuple(A.shape)} + B {tuple(B.shape)} = {A.numel() + B.numel():,} pesos entrenables "
      f"({(A.numel() + B.numel()) / full:.1%} de W)")

Adaptador cargado desde Bellido/qwen2.5-0.5b-cafeterias-uam-lora
trainable params: 0 || all params: 502,830,976 || trainable%: 0.0000

q_proj capa 0: W (401408, 1) = 401,408 pesos (congelados)
  A (16, 896) + B (896, 16) = 28,672 pesos entrenables (7.1% de W)


## 6. Preparar los datos y entrenar

Cada ejemplo se convierte con la plantilla de chat (system + pregunta + respuesta). Las etiquetas del **prompt** se ponen a `-100`.
La idea es enseñar al modelo a generar únicamente la respuesta, sin penalizarlo ni evaluar cómo escribe la pregunta.

La idea es sacar el prompt y el prompt + repuesta y solo evaluar la diferencia, es decir, únicamente la respuesta

In [36]:
if ENTRENAR: 
    MAX_LEN = 192

    def tokenize(example):
        prompt = tokenizer.apply_chat_template(build_messages(example["question"]), tokenize=False, add_generation_prompt=True) # (NO incluye la respuesta) Generamos el prompt de entrada para el modelo, incluye la pregunta del usuario y el rol del sistema
        full = tokenizer.apply_chat_template(build_messages(example["question"], example["answer"]), tokenize=False) # Generamos el prompt completo (la conversación completa), incluye la respuesta del asistente
        assert full.startswith(prompt) # Comprueba que full empieza exactamente igual que prompt
        
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"] # Se obtienen los ID del prompt
        full_ids = tokenizer(full, add_special_tokens=False)["input_ids"][:MAX_LEN] # Se optienen los ID del prompt completo
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):] # No calcula la pérdida para los tokens de la pregunta, solo para los de la respuesta
        return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels[:MAX_LEN]}


    # Pasa cada fila del dataset por la función tokenize y elimina las columnas originales (question, answer) para quedarse solo con input_ids, attention_mask y labels
    train_tok = train_ds.map(tokenize, remove_columns=train_ds.column_names) # dataset tokenizado para entrenamiento
    eval_tok = eval_ds.map(tokenize, remove_columns=eval_ds.column_names) # dataset tokenizado para evaluación

    ex = train_tok[0]
    print("Texto completo :", tokenizer.decode(ex["input_ids"]))
    print("Solo se aprende:", tokenizer.decode([t for t in ex["labels"] if t != -100]))

In [37]:
if ENTRENAR:
    from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

    model.config.use_cache = False   # la caché de generación no se usa al entrenar

    args = TrainingArguments( # Parámetros de entrenamiento que me han funcionado
        output_dir=ADAPTER_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        fp16=(DEVICE == "cuda" and DTYPE == torch.float16),
        bf16=(DEVICE == "cuda" and DTYPE == torch.bfloat16),
        dataloader_pin_memory=(DEVICE == "cuda"),
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok, # Pasamos el dataset tokenizado para entrenamiento
        eval_dataset=eval_tok, # Pasamos el dataset tokenizado para evaluación
        data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100), 
    )
    trainer.train()

## 7. Prueba *después* de entrenar

In [38]:
model.config.use_cache = True
model.eval()

answers_lora = {q: ask(model, q) for q in TEST_QUESTIONS}

pd.set_option("display.max_colwidth", None)
comparison = pd.DataFrame({
    "Pregunta": TEST_QUESTIONS,
    "Antes (base)": [answers_base[q] for q in TEST_QUESTIONS],
    "Después (LoRA)": [answers_lora[q] for q in TEST_QUESTIONS],
})
display(comparison)

,Pregunta,Antes (base),Después (LoRA)
0,¿Cuánto cuesta un café solo?,"El costo del café en una cafetería de la Universidad Autónoma de Madrid puede variar dependiendo de varios factores, como el tipo de café (especialmente si es de origen local), la ubicación y el tiempo","El café solo cuesta 1,10 €."
1,¿Qué precio tiene el bocadillo de lomo?,El precio del bocadillo de lomo en la cafetería de la Universidad Autónoma de Madrid es de 15 euros.,"El bocadillo de lomo cuesta 2,35 €."
2,¿Cuánto vale el menú autoservicio?,"El menú autoservicio en la Universidad Autónoma de Madrid es gratuito para los estudiantes, pero se puede pagar con tarjetas de crédito o débito.","El menú autoservicio cuesta 6,75 €."
3,¿Cuánto cuesta la pizza entera?,"La pizzas entera en la Universidad Autónoma de Madrid puede variar dependiendo del tipo de pizza, el tamaño y los ingredientes adicionales que se incluyan. Sin embargo, generalmente, una pizza completa puede costarse","La pizza cuesta 9,80 € (1 chino, 2 raciones de pizza y 1 sándwich)."
4,¿Cuánto cuesta el bono de 10 comidas?,"El bono de 10 comidas en una cafetería de la UAM varía dependiendo del lugar y el tipo de servicio. En general, puede variar entre €5 a €20 por comida. Por","El bono de 10 comidas cuesta 60,75 €."
5,¿Qué opciones vegetarianas hay en la cafetería?,"En la cafetería de la Universidad Autónoma de Madrid, existen varias opciones vegetarianas disponibles:\n\n1. Sándwiches vegetarianos: Ofrece una variedad de sándwiches vegetarianos como quinoa, tofu","La cafeteria vegetal cuesta 0,80 €."


In [39]:
if DATASET_ID == "Bellido/cafeterias_uam_2026":
    metrics["lora"] = {
        "train": price_accuracy(model, probe_train),
        "eval": price_accuracy(model, probe_eval),
    }
    display(pd.DataFrame(metrics).T.rename(columns={"train": "Preguntas vistas (train)", "eval": "Preguntas reservadas (eval)"}))

,Preguntas vistas (train),Preguntas reservadas (eval)
base,0.0,0.00
lora,1.0,0.95


### Activar y desactivar el adaptador

Con LoRA el modelo base sigue intacto. Podemos apagar el adaptador y ver como cambia la respuesta

In [40]:
question = "¿Cuánto cuesta la hamburguesa con queso, tomate, cebolla y lechuga?"

with model.disable_adapter():
    print("Sin adaptador:", ask(model, question))
print("Con adaptador:", ask(model, question))

Sin adaptador: La hamburguesa con queso, tomate, cebolla y lechuga costará aproximadamente 15-20 euros en la mayoría de los mercados locales. Sin embargo, esto puede variar dependiendo del
Con adaptador: La hamburguesa con queso, tomate, cebolla y lechuga cuesta 2,80 €.


## 8. Guardar el adaptador

Solo se guardan las matrices `A` y `B`: unos pocos MB en lugar de los GB del modelo. Para usarlo se carga el modelo base y se le aplica el adaptador encima (`PeftModel.from_pretrained`), o se fusiona con `merge_and_unload()` para obtener un modelo normal.

In [41]:
import os

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

size_mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)
              if f.startswith("adapter_model")) / 1e6
print(f"Adaptador guardado en '{ADAPTER_DIR}/' ({size_mb:.1f} MB)")

Adaptador guardado en 'lora-cafeterias-uam/' (35.2 MB)


## 9. Demo interactiva

Podemos cargar el modelo del directorio directamente

In [44]:
# model = AutoModelForCausalLM.from_pretrained(ADAPTER_DIR).to(DEVICE)
# print("Modelo cargado desde el directorio:", ADAPTER_DIR)

Loading weights: 100%|██████████| 336/336 [00:00<00:00, 6811.53it/s]


Modelo cargado desde el directorio: lora-cafeterias-uam


In [45]:
def demo():
    while True:
        q = input("Pregunta (vacío para salir): ").strip()
        if not q:
            break
        print(q,"->", ask(model, q), "\n")
demo()

ihosx -> El ihosx es una copa de vino blanco de origen autóctono. 

hola -> Hola! ¿Cómo estás? 

ofgkjshfhbnsodn -> El ofgkjshfhbnsodn es una frase de texto. 

hola -> Hola! ¿Cómo estás? 



## 10. Subir el modelo a HuggingFace

In [43]:
# from huggingface_hub import notebook_login
# USUARIO = "ejemplo"
# ADAPTER_DIR = "qwen2.5-0.5b-cafeterias-uam-lora"
# URI = f"{USUARIO}/{ADAPTER_DIR}"

# notebook_login()
# model.push_to_hub(URI)
# tokenizer.push_to_hub(URI)